[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

# def causal_attention(Q, K, V):
#     """
#     Compute the causal attention of the given query, key, and value tensors.

#     Args:
#         Q: A tensor of shape (batch_size, seq_len, d_k) representing the query.
#         K: A tensor of shape (batch_size, seq_len, d_k) representing the key.
#         V: A tensor of shape (batch_size, seq_len, d_v) representing the value.
#     Returns:
#         A tensor of shape (batch_size, seq_len, d_v) representing the output of the causal attention.
#     """
#     # Step 1: Compute the attention scores
#     d_k = Q.size(-1)
#     scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

#     # Step 2: Create a causal mask to prevent attending to future positions
#     seq_len = Q.size(1)
#     mask = torch.tril(torch.ones(seq_len, seq_len)).to(Q.device)  # (seq_len, seq_len)
#     scores = scores.masked_fill(mask == 0, float('-inf'))  # Mask out future positions

#     # Step 3: Apply softmax to get attention weights
#     attn_weights = torch.softmax(scores, dim=-1)  # (batch_size, seq_len, seq_len)

#     # Step 4: Compute the output by multiplying attention weights with the value tensor
#     output = torch.matmul(attn_weights, V)  # (batch_size, seq_len, d_v)

#     return output

In [4]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):
    # Q: (batch_size, seq_len, d_k)
    # K: (batch_size, seq_len, d_k)
    # V: (batch_size, seq_len, d_k)
    _, seq_len, d_k = Q.shape
    scores_scaled = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (batch_size, seq_len, d_k) x (batch_size, d_k, seq_len) -> (batch_size, seq_len, seq_len)
    masks = torch.tril(torch.ones(seq_len, seq_len))
    scores_scaled = scores_scaled.masked_fill(masks == 0, float('-inf'))
    attn_weights = torch.softmax(scores_scaled, dim=-1)
    output = attn_weights @ V
    return output

In [5]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [6]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (1.1ms)
  ✅ [2/4] Future tokens don't affect past (1.8ms)
  ✅ [3/4] First position only sees itself (0.7ms)
  ✅ [4/4] Gradient flow (19.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (22.8ms total)
  Progress saved. Run status() to see your dashboard.

